In [ ]:
# PATH CONFIGURATION ONLY: edit these locations for your runtime.
# In Colab, upload the archive into WORK_ROOT, or set DATASET_ZIP explicitly.
# Configure absolute locations before running subsequent cells in order.
from pathlib import Path
import os

WORK_ROOT = Path.cwd().resolve()
DATASET_ZIP = WORK_ROOT / "fr1_xyz_lightglue.zip"
DATASET_EXTRACT_ROOT = WORK_ROOT / "fr1_xyz_lightglue"
RESULTS_ROOT = WORK_ROOT / "experiment18I_shared_backend"
LIGHTGLUE_DIR = WORK_ROOT / "LightGlue"
RESULT_ARCHIVE_BASE = WORK_ROOT / "experiment18I_shared_backend"
VERIFICATION_ROOT = WORK_ROOT
# Search only the configured sequence extraction tree.
DATASET_SEARCH_ROOT = DATASET_EXTRACT_ROOT
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
VERIFICATION_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!git clone -q https://github.com/cvg/LightGlue.git "{LIGHTGLUE_DIR}"
%cd {LIGHTGLUE_DIR}
!pip install -q -e .
os.chdir(WORK_ROOT)


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

for item in os.listdir(str(WORK_ROOT)):
    print(item)

In [ ]:
import os
print(os.listdir(str(WORK_ROOT)))

In [ ]:
import zipfile
import os

zip_path = str(DATASET_ZIP)
extract_dir = str(DATASET_EXTRACT_ROOT)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Extracted to:", extract_dir)

for root, dirs, files in os.walk(extract_dir):
    if os.path.basename(root) == "rgb":
        pngs = [f for f in files if f.lower().endswith(".png")]
        if len(pngs) > 0:
            print("Found rgb folder:", root)
            print("PNG count:", len(pngs))
            break

In [ ]:
import os
import numpy as np
from PIL import Image

# ------------------------------------------------------------
# FIND RGB FOLDER
# ------------------------------------------------------------

rgb_dir = None

for root, dirs, files in os.walk(str(DATASET_EXTRACT_ROOT)):
    if os.path.basename(root) == "rgb":
        pngs = [f for f in files if f.lower().endswith(".png")]
        if len(pngs) > 100:
            rgb_dir = root
            break

if rgb_dir is None:
    raise FileNotFoundError("rgb folder not found")

print("Using rgb folder:")
print(rgb_dir)

# ------------------------------------------------------------
# SELECT SAME IMAGE 100
# ------------------------------------------------------------

files = sorted([
    f for f in os.listdir(rgb_dir)
    if f.lower().endswith(".png")
])

print("PNG count:", len(files))

img_name = files[99]   # Python index 99 = MATLAB image 100
img_path = os.path.join(rgb_dir, img_name)

print("Using image:")
print(img_path)

img = np.array(
    Image.open(img_path).convert("RGB")
)

print("Original shape:", img.shape)

# ------------------------------------------------------------
# MATCHED PWM SETTINGS
# ------------------------------------------------------------

H = img.shape[0]
Ns = 100

freq_hz = 100
exposure_s = 0.001
duty = 0.25
mod_depth = 0.90

phase_offset = 0.25
readout_ms = 20

readout_s = readout_ms / 1000.0
row_time = readout_s / H

row_gain = np.zeros(H, dtype=np.float64)

for r in range(H):

    exposure_offsets = np.linspace(
        0.0,
        exposure_s,
        Ns
    )

    t = r * row_time + exposure_offsets

    phi = np.mod(
        freq_hz * t + phase_offset,
        1.0
    )

    led_on = phi < duty

    instantaneous_gain = (
        (1.0 - mod_depth)
        + mod_depth * led_on.astype(float)
    )

    row_gain[r] = np.mean(instantaneous_gain)

# ------------------------------------------------------------
# APPLY GAIN
# ------------------------------------------------------------

perturbed = (
    img.astype(np.float64)
    * row_gain[:, None, None]
)

perturbed = np.clip(
    np.rint(perturbed),
    0,
    255
).astype(np.uint8)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

out_img = str(VERIFICATION_ROOT / "perturbed_python.png")

Image.fromarray(perturbed).save(out_img)

print("\nSaved:")
print(out_img)
print("Shape:", perturbed.shape)

In [ ]:
with zipfile.ZipFile(DATASET_ZIP, "r") as z:
    z.extractall(DATASET_EXTRACT_ROOT)


In [ ]:
%cd {WORK_ROOT}


In [ ]:
import os
from pathlib import Path

BASE = DATASET_SEARCH_ROOT

# Find groundtruth.txt automatically
gt_candidates = list(BASE.rglob("groundtruth.txt"))

if len(gt_candidates) == 0:
    raise FileNotFoundError(f"groundtruth.txt was not found under {DATASET_SEARCH_ROOT}")

DATASET_ROOT = gt_candidates[0].parent
RGB_DIR = DATASET_ROOT / "rgb"
GT_FILE = DATASET_ROOT / "groundtruth.txt"

print("Dataset root:", DATASET_ROOT)
print("RGB folder:", RGB_DIR)
print("Ground truth:", GT_FILE)

if not RGB_DIR.exists():
    raise FileNotFoundError(f"RGB folder not found: {RGB_DIR}")

In [ ]:
import os

if not os.path.exists(str(LIGHTGLUE_DIR)):
    !git clone -q https://github.com/cvg/LightGlue.git "{LIGHTGLUE_DIR}"

%cd {LIGHTGLUE_DIR}
!pip install -q -e .

In [ ]:
import math
import cv2
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from lightglue import LightGlue, SuperPoint
from lightglue.utils import rbd

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
rgb_files = sorted(
    list(RGB_DIR.glob("*.png")),
    key=lambda p: float(p.stem)
)

print("RGB images:", len(rgb_files))
print("First:", rgb_files[0])
print("Last :", rgb_files[-1])

assert len(rgb_files) == 798, \
    f"Expected 798 RGB images, found {len(rgb_files)}"

In [ ]:
START_INDEX = 99       # Python index = MATLAB image 100
GAP = 3
N_PAIRS = 50

PHASES = [
    0.0,
    0.125,
    0.25,
    0.5,
    0.875
]

READOUT_MS = [
    5,
    10,
    15,
    20,
    25,
    30
]

PWM_FREQ_HZ = 100.0
EXPOSURE_MS = 1.0
DUTY_CYCLE = 0.25
MODULATION_DEPTH = 0.90

# IMPORTANT: same numerical exposure integration used previously
NUM_SAMPLES = 100

print("Total PWM evaluations:",
      N_PAIRS * len(PHASES) * len(READOUT_MS))

In [ ]:
fx = 517.3
fy = 516.5
cx = 318.6
cy = 255.3

K = np.array([
    [fx, 0.0, cx],
    [0.0, fy, cy],
    [0.0, 0.0, 1.0]
], dtype=np.float64)

print(K)

In [ ]:
gt_data = np.loadtxt(
    GT_FILE,
    comments="#"
)

gt_times = gt_data[:, 0]

print("Ground-truth poses:", len(gt_data))

assert len(gt_data) > 0

In [ ]:
def load_rgb_image(path):
    return np.array(
        Image.open(path).convert("RGB")
    )


def numpy_to_torch_image(img):

    arr = np.asarray(img)

    tensor = torch.from_numpy(
        arr.copy()
    ).permute(2, 0, 1).float() / 255.0

    return tensor.to(device)


def quaternion_to_rotation_matrix(qx, qy, qz, qw):

    q = np.array(
        [qx, qy, qz, qw],
        dtype=np.float64
    )

    q /= np.linalg.norm(q)

    qx, qy, qz, qw = q

    return np.array([

        [
            1 - 2*(qy*qy + qz*qz),
            2*(qx*qy - qz*qw),
            2*(qx*qz + qy*qw)
        ],

        [
            2*(qx*qy + qz*qw),
            1 - 2*(qx*qx + qz*qz),
            2*(qy*qz - qx*qw)
        ],

        [
            2*(qx*qz - qy*qw),
            2*(qy*qz + qx*qw),
            1 - 2*(qx*qx + qy*qy)
        ]

    ], dtype=np.float64)

In [ ]:
def nearest_gt_pose(timestamp):

    idx = np.argmin(
        np.abs(gt_times - timestamp)
    )

    row = gt_data[idx]

    t = row[1:4].astype(np.float64)

    qx, qy, qz, qw = row[4:8]

    R = quaternion_to_rotation_matrix(
        qx, qy, qz, qw
    )

    return R, t, gt_times[idx]


def get_pair_gt_pose(i, j):

    timestamp1 = float(
        rgb_files[i].stem
    )

    timestamp2 = float(
        rgb_files[j].stem
    )

    R1, t1, gt_time1 = nearest_gt_pose(
        timestamp1
    )

    R2, t2, gt_time2 = nearest_gt_pose(
        timestamp2
    )

    # Camera 2 orientation relative to Camera 1
    R_relative = R1.T @ R2

    # Camera 2 location expressed in Camera 1 coordinates
    t_relative = R1.T @ (
        t2 - t1
    )

    baseline = np.linalg.norm(
        t_relative
    )

    if baseline < 1e-12:
        raise ValueError(
            "GT translation baseline is effectively zero."
        )

    t_direction = (
        t_relative / baseline
    )

    return {

        "R_orientation":
            R_relative,

        "t_direction":
            t_direction,

        "baseline_m":
            baseline,

        "rgb_time1":
            timestamp1,

        "rgb_time2":
            timestamp2,

        "gt_time1":
            gt_time1,

        "gt_time2":
            gt_time2,

        "assoc_diff1_s":
            abs(timestamp1 - gt_time1),

        "assoc_diff2_s":
            abs(timestamp2 - gt_time2)
    }

In [ ]:
def rotation_error_deg(R_est, R_gt):

    R_diff = (
        R_est.T @ R_gt
    )

    value = (
        np.trace(R_diff) - 1.0
    ) / 2.0

    value = np.clip(
        value,
        -1.0,
        1.0
    )

    return np.degrees(
        np.arccos(value)
    )


def vector_angle_deg(v1, v2):

    v1 = np.asarray(
        v1,
        dtype=np.float64
    )

    v2 = np.asarray(
        v2,
        dtype=np.float64
    )

    v1 /= np.linalg.norm(v1)
    v2 /= np.linalg.norm(v2)

    value = np.clip(
        np.dot(v1, v2),
        -1.0,
        1.0
    )

    return np.degrees(
        np.arccos(value)
    )

In [ ]:
def simulate_pwm_rolling_shutter(
    image,
    pwm_freq_hz,
    exposure_ms,
    duty_cycle,
    modulation_depth,
    readout_ms,
    phase_offset,
    num_samples=100
):

    img = np.asarray(
        image
    ).astype(np.float64)

    H = img.shape[0]

    readout_s = (
        readout_ms / 1000.0
    )

    exposure_s = (
        exposure_ms / 1000.0
    )

    row_time = (
        readout_s / H
    )

    # Same controlled numerical exposure integration
    exposure_offsets = np.linspace(
        0.0,
        exposure_s,
        num_samples
    )

    row_gain = np.zeros(
        H,
        dtype=np.float64
    )

    for r in range(H):

        row_start = (
            r * row_time
        )

        sample_times = (
            row_start +
            exposure_offsets
        )

        pwm_phase = np.mod(
            sample_times *
            pwm_freq_hz +
            phase_offset,
            1.0
        )

        led_on = (
            pwm_phase <
            duty_cycle
        )

        instantaneous_gain = np.where(
            led_on,
            1.0,
            1.0 - modulation_depth
        )

        row_gain[r] = np.mean(
            instantaneous_gain
        )

    perturbed = (
        img *
        row_gain[:, None, None]
    )

    perturbed = np.clip(
        np.rint(perturbed),
        0,
        255
    ).astype(np.uint8)

    return perturbed

In [ ]:
extractor = SuperPoint(
    max_num_keypoints=2048
).eval().to(device)

matcher = LightGlue(
    features="superpoint"
).eval().to(device)

print("SuperPoint + LightGlue ready.")

In [ ]:
def evaluate_pair(
    img1,
    img2,
    gt_pose,
    pair_id=None,
    phase=None,
    readout_ms=None,
    condition_type="PWM",
    match_storage=None
):

    image0 = numpy_to_torch_image(
        img1
    )

    image1 = numpy_to_torch_image(
        img2
    )

    with torch.inference_mode():

        feats0 = extractor.extract(
            image0,
            resize=None
        )

        feats1 = extractor.extract(
            image1,
            resize=None
        )

        matches01 = matcher({
            "image0": feats0,
            "image1": feats1
        })

    feats0_r, feats1_r, matches_r = [
        rbd(x)
        for x in [
            feats0,
            feats1,
            matches01
        ]
    ]

    kp0 = (
        feats0_r["keypoints"]
        .detach()
        .cpu()
        .numpy()
    )

    kp1 = (
        feats1_r["keypoints"]
        .detach()
        .cpu()
        .numpy()
    )

    matches = (
        matches_r["matches"]
        .detach()
        .cpu()
        .numpy()
    )

    n_kp1 = len(kp0)
    n_kp2 = len(kp1)
    n_matches = len(matches)

    result = {

        "Keypoints1": n_kp1,
        "Keypoints2": n_kp2,
        "Matches": n_matches,

        "FundamentalInliers": 0,

        "FundamentalInlierRatioPercent":
            0.0,

        "EssentialInliers": 0,

        "PoseInliers": 0,

        "PoseCheiralityFraction":
            0.0,

        "PoseRecovered": False,

        "RotationErrorDeg":
            np.nan,

        "TranslationRawErrorDeg":
            np.nan,

        "TranslationSignInvariantErrorDeg":
            np.nan
    }

    if n_matches > 0:

        points0 = kp0[
            matches[:, 0]
        ].astype(np.float64)

        points1 = kp1[
            matches[:, 1]
        ].astype(np.float64)

        if match_storage is not None:

            for m in range(
                n_matches
            ):

                match_storage.append({

                    "ConditionType":
                        condition_type,

                    "PairID":
                        pair_id,

                    "Phase":
                        np.nan
                        if phase is None
                        else float(phase),

                    "ReadoutMs":
                        np.nan
                        if readout_ms is None
                        else float(readout_ms),

                    "MatchID":
                        m + 1,

                    "X1":
                        float(points0[m, 0]),

                    "Y1":
                        float(points0[m, 1]),

                    "X2":
                        float(points1[m, 0]),

                    "Y2":
                        float(points1[m, 1])
                })

    else:

        points0 = np.empty(
            (0, 2),
            dtype=np.float64
        )

        points1 = np.empty(
            (0, 2),
            dtype=np.float64
        )

    if n_matches < 8:
        return result

    # Existing OpenCV geometry retained only as diagnostic.
    # Final fair comparison will be recomputed in MATLAB.

    F, maskF = cv2.findFundamentalMat(
        points0,
        points1,
        cv2.FM_RANSAC,
        1.5,
        0.999,
        2000
    )

    if (
        F is not None
        and maskF is not None
    ):

        maskF_bool = (
            maskF.ravel() > 0
        )

        nF = int(
            np.sum(maskF_bool)
        )

        result[
            "FundamentalInliers"
        ] = nF

        result[
            "FundamentalInlierRatioPercent"
        ] = (
            100.0 *
            nF /
            n_matches
        )

    try:

        E, maskE = cv2.findEssentialMat(
            points0,
            points1,
            K,
            method=cv2.RANSAC,
            prob=0.999,
            threshold=1.5,
            maxIters=2000
        )

    except cv2.error:

        return result

    if E is None:
        return result

    if E.shape != (3, 3):

        if (
            len(E.shape) == 2
            and E.shape[1] == 3
            and E.shape[0] >= 3
        ):

            E = E[:3, :]

        else:

            return result

    if maskE is not None:

        maskE_bool = (
            maskE.ravel() > 0
        )

        points0_E = points0[
            maskE_bool
        ]

        points1_E = points1[
            maskE_bool
        ]

    else:

        points0_E = points0
        points1_E = points1

    n_E = len(
        points0_E
    )

    result[
        "EssentialInliers"
    ] = n_E

    if n_E < 8:
        return result

    try:

        pose_inliers, R_cv, t_cv, pose_mask = \
            cv2.recoverPose(
                E,
                points0_E,
                points1_E,
                K
            )

    except cv2.error:

        return result

    pose_inliers = int(
        pose_inliers
    )

    result[
        "PoseInliers"
    ] = pose_inliers

    result[
        "PoseCheiralityFraction"
    ] = (
        pose_inliers /
        n_E
    )

    if pose_inliers < 8:
        return result

    # OpenCV x2 = R*x1 + t
    # Convert to camera-2 orientation/location in camera-1 frame.

    R_est_orientation = (
        R_cv.T
    )

    t_est_location = (
        -R_cv.T @
        t_cv.reshape(3)
    )

    norm_t = np.linalg.norm(
        t_est_location
    )

    if norm_t < 1e-12:
        return result

    t_est_location /= norm_t

    rot_err = rotation_error_deg(
        R_est_orientation,
        gt_pose["R_orientation"]
    )

    trans_raw = vector_angle_deg(
        t_est_location,
        gt_pose["t_direction"]
    )

    trans_sign_inv = min(
        trans_raw,
        180.0 - trans_raw
    )

    result[
        "PoseRecovered"
    ] = True

    result[
        "RotationErrorDeg"
    ] = rot_err

    result[
        "TranslationRawErrorDeg"
    ] = trans_raw

    result[
        "TranslationSignInvariantErrorDeg"
    ] = trans_sign_inv

    return result

In [ ]:
OUT_DIR = Path(
    RESULTS_ROOT
)

OUT_DIR.mkdir(
    exist_ok=True
)

CLEAN_MATCH_CSV = (
    OUT_DIR /
    "experiment18I_LightGlue_clean_matches.csv"
)

PWM_MATCH_CSV = (
    OUT_DIR /
    "experiment18I_LightGlue_PWM_matches.csv"
)

CLEAN_SUMMARY_CSV = (
    OUT_DIR /
    "experiment18I_LightGlue_clean_summary.csv"
)

PWM_SUMMARY_CSV = (
    OUT_DIR /
    "experiment18I_LightGlue_PWM_summary.csv"
)

CHECK_CSV = (
    OUT_DIR /
    "experiment18I_export_integrity_check.csv"
)

print(OUT_DIR)

In [ ]:
import os

for path in [
    CLEAN_MATCH_CSV,
    PWM_MATCH_CSV,
    CLEAN_SUMMARY_CSV,
    PWM_SUMMARY_CSV,
    CHECK_CSV
]:
    if os.path.exists(path):
        os.remove(path)

print("Old Exp18I outputs cleared.")

In [ ]:
def append_matches_to_csv(
    rows,
    file_path
):

    if len(rows) == 0:
        return

    df = pd.DataFrame(
        rows
    )

    df.to_csv(
        file_path,
        mode="a",
        index=False,
        header=not file_path.exists()
    )

In [ ]:
def make_summary_row(
    result,
    pair_id,
    idx1,
    idx2,
    phase,
    readout_ms,
    condition_type
):

    return {

        "ConditionType":
            condition_type,

        "PairID":
            pair_id,

        "ImageIndex1":
            idx1,

        "ImageIndex2":
            idx2,

        "Phase":
            phase,

        "ReadoutMs":
            readout_ms,

        "Keypoints1":
            result["Keypoints1"],

        "Keypoints2":
            result["Keypoints2"],

        "Matches":
            result["Matches"],

        "FundamentalInliers":
            result["FundamentalInliers"],

        "FundamentalInlierRatioPercent":
            result[
                "FundamentalInlierRatioPercent"
            ],

        "EssentialInliers":
            result["EssentialInliers"],

        "PoseInliers":
            result["PoseInliers"],

        "PoseCheiralityFraction":
            result[
                "PoseCheiralityFraction"
            ],

        "PoseRecovered":
            result["PoseRecovered"],

        "RotationErrorDeg":
            result["RotationErrorDeg"],

        "TranslationRawErrorDeg":
            result[
                "TranslationRawErrorDeg"
            ],

        "TranslationSignInvariantErrorDeg":
            result[
                "TranslationSignInvariantErrorDeg"
            ]
    }

In [ ]:
print(
    "Number of RGB frames:",
    len(rgb_files)
)

print("\nFirst selected pair:")

print(
    rgb_files[
        START_INDEX
    ]
)

print(
    rgb_files[
        START_INDEX + GAP
    ]
)

last_idx = (
    START_INDEX +
    N_PAIRS -
    1
)

print("\nLast selected pair:")

print(
    rgb_files[
        last_idx
    ]
)

print(
    rgb_files[
        last_idx + GAP
    ]
)

In [ ]:
i = START_INDEX
j = i + GAP

img1 = load_rgb_image(
    rgb_files[i]
)

img2 = load_rgb_image(
    rgb_files[j]
)

gt_pose = get_pair_gt_pose(
    i,
    j
)

test_matches = []

test_result = evaluate_pair(
    img1,
    img2,
    gt_pose,

    pair_id=1,

    phase=None,

    readout_ms=None,

    condition_type="CLEAN",

    match_storage=test_matches
)

print(test_result)

print(
    "\nRaw matches exported:",
    len(test_matches)
)

print(
    pd.DataFrame(
        test_matches
    ).head()
)

In [ ]:
clean_summary_rows = []

for pair_num in tqdm(
    range(N_PAIRS),
    desc="CLEAN pairs"
):

    i = (
        START_INDEX +
        pair_num
    )

    j = (
        i + GAP
    )

    img1 = load_rgb_image(
        rgb_files[i]
    )

    img2 = load_rgb_image(
        rgb_files[j]
    )

    gt_pose = get_pair_gt_pose(
        i,
        j
    )

    case_matches = []

    result = evaluate_pair(
        img1,
        img2,
        gt_pose,

        pair_id=pair_num + 1,

        phase=None,

        readout_ms=None,

        condition_type="CLEAN",

        match_storage=case_matches
    )

    append_matches_to_csv(
        case_matches,
        CLEAN_MATCH_CSV
    )

    clean_summary_rows.append(
        make_summary_row(
            result,
            pair_num + 1,
            i,
            j,
            np.nan,
            np.nan,
            "CLEAN"
        )
    )

pd.DataFrame(
    clean_summary_rows
).to_csv(
    CLEAN_SUMMARY_CSV,
    index=False
)

print(
    "Clean export complete:",
    len(clean_summary_rows),
    "pairs"
)

In [ ]:
clean_summary = pd.read_csv(
    CLEAN_SUMMARY_CSV
)

clean_matches = pd.read_csv(
    CLEAN_MATCH_CSV
)

print(
    "Summary rows:",
    len(clean_summary)
)

print(
    "Raw clean matches:",
    len(clean_matches)
)

print(
    clean_summary[
        [
            "PairID",
            "Matches",
            "PoseRecovered",
            "RotationErrorDeg",
            "TranslationRawErrorDeg"
        ]
    ].head(10)
)

In [ ]:
i = START_INDEX
j = i + GAP

img1_clean = load_rgb_image(
    rgb_files[i]
)

img2_clean = load_rgb_image(
    rgb_files[j]
)

phase = 0.25
readout_ms = 20

img1_pwm = simulate_pwm_rolling_shutter(
    img1_clean,

    pwm_freq_hz=
        PWM_FREQ_HZ,

    exposure_ms=
        EXPOSURE_MS,

    duty_cycle=
        DUTY_CYCLE,

    modulation_depth=
        MODULATION_DEPTH,

    readout_ms=
        readout_ms,

    phase_offset=
        0.0,

    num_samples=
        NUM_SAMPLES
)

img2_pwm = simulate_pwm_rolling_shutter(
    img2_clean,

    pwm_freq_hz=
        PWM_FREQ_HZ,

    exposure_ms=
        EXPOSURE_MS,

    duty_cycle=
        DUTY_CYCLE,

    modulation_depth=
        MODULATION_DEPTH,

    readout_ms=
        readout_ms,

    phase_offset=
        phase,

    num_samples=
        NUM_SAMPLES
)

gt_pose = get_pair_gt_pose(
    i,
    j
)

test_pwm_matches = []

test_pwm_result = evaluate_pair(
    img1_pwm,
    img2_pwm,
    gt_pose,

    pair_id=1,

    phase=phase,

    readout_ms=readout_ms,

    condition_type="PWM",

    match_storage=
        test_pwm_matches
)

print(test_pwm_result)

print(
    "Raw PWM matches:",
    len(test_pwm_matches)
)

In [ ]:
pwm_summary_rows = []

total_cases = (
    N_PAIRS *
    len(PHASES) *
    len(READOUT_MS)
)

pbar = tqdm(
    total=total_cases,
    desc="PWM LightGlue"
)

for pair_num in range(
    N_PAIRS
):

    i = (
        START_INDEX +
        pair_num
    )

    j = (
        i + GAP
    )

    img1_clean = load_rgb_image(
        rgb_files[i]
    )

    img2_clean = load_rgb_image(
        rgb_files[j]
    )

    gt_pose = get_pair_gt_pose(
        i,
        j
    )

    for readout_ms in READOUT_MS:

        for phase in PHASES:

            # Frame 1:
            # controlled reference phase = 0

            img1_pwm = simulate_pwm_rolling_shutter(
                img1_clean,

                pwm_freq_hz=
                    PWM_FREQ_HZ,

                exposure_ms=
                    EXPOSURE_MS,

                duty_cycle=
                    DUTY_CYCLE,

                modulation_depth=
                    MODULATION_DEPTH,

                readout_ms=
                    readout_ms,

                phase_offset=
                    0.0,

                num_samples=
                    NUM_SAMPLES
            )

            # Frame 2:
            # explicit relative phase offset

            img2_pwm = simulate_pwm_rolling_shutter(
                img2_clean,

                pwm_freq_hz=
                    PWM_FREQ_HZ,

                exposure_ms=
                    EXPOSURE_MS,

                duty_cycle=
                    DUTY_CYCLE,

                modulation_depth=
                    MODULATION_DEPTH,

                readout_ms=
                    readout_ms,

                phase_offset=
                    phase,

                num_samples=
                    NUM_SAMPLES
            )

            case_matches = []

            result = evaluate_pair(
                img1_pwm,
                img2_pwm,
                gt_pose,

                pair_id=
                    pair_num + 1,

                phase=
                    phase,

                readout_ms=
                    readout_ms,

                condition_type=
                    "PWM",

                match_storage=
                    case_matches
            )

            append_matches_to_csv(
                case_matches,
                PWM_MATCH_CSV
            )

            pwm_summary_rows.append(
                make_summary_row(
                    result,

                    pair_num + 1,

                    i,
                    j,

                    phase,

                    readout_ms,

                    "PWM"
                )
            )

            pbar.update(1)

pbar.close()

pd.DataFrame(
    pwm_summary_rows
).to_csv(
    PWM_SUMMARY_CSV,
    index=False
)

print(
    "PWM export complete:",
    len(pwm_summary_rows),
    "conditions"
)

In [ ]:
pwm_summary = pd.read_csv(
    PWM_SUMMARY_CSV
)

pwm_matches = pd.read_csv(
    PWM_MATCH_CSV
)

clean_summary = pd.read_csv(
    CLEAN_SUMMARY_CSV
)

clean_matches = pd.read_csv(
    CLEAN_MATCH_CSV
)

print("==============================")
print("EXPERIMENT 18I CHECK")
print("==============================")

print(
    "Clean summary rows:",
    len(clean_summary)
)

print(
    "PWM summary rows:",
    len(pwm_summary)
)

print(
    "Expected PWM rows:",
    1500
)

print(
    "Clean raw correspondence rows:",
    len(clean_matches)
)

print(
    "PWM raw correspondence rows:",
    len(pwm_matches)
)

print(
    "Unique pairs:",
    pwm_summary[
        "PairID"
    ].nunique()
)

print(
    "Phases:",
    sorted(
        pwm_summary[
            "Phase"
        ].unique()
    )
)

print(
    "Readouts:",
    sorted(
        pwm_summary[
            "ReadoutMs"
        ].unique()
    )
)

In [ ]:
coordinate_counts = (
    pwm_matches
    .groupby(
        [
            "PairID",
            "Phase",
            "ReadoutMs"
        ]
    )
    .size()
    .reset_index(
        name="ExportedRawMatches"
    )
)

check = pwm_summary.merge(
    coordinate_counts,

    on=[
        "PairID",
        "Phase",
        "ReadoutMs"
    ],

    how="left"
)

check[
    "ExportedRawMatches"
] = (
    check[
        "ExportedRawMatches"
    ]
    .fillna(0)
    .astype(int)
)

check[
    "CountDifference"
] = (
    check["Matches"] -
    check["ExportedRawMatches"]
)

print(
    check[
        "CountDifference"
    ].value_counts()
)

assert (
    check[
        "CountDifference"
    ] == 0
).all()

print(
    "\nPASS: all raw LightGlue matches were exported."
)

check.to_csv(
    CHECK_CSV,
    index=False
)

In [ ]:
import shutil

ZIP_BASE = (
    str(RESULT_ARCHIVE_BASE)
)

shutil.make_archive(
    ZIP_BASE,
    "zip",
    str(OUT_DIR)
)

print(
    "Created:",
    ZIP_BASE + ".zip"
)

In [ ]:
from google.colab import files

files.download(
    str(RESULT_ARCHIVE_BASE) + ".zip"
)

In [ ]:
import numpy as np
import pandas as pd

H = 480
Ns = 100

freq_hz = 100
exposure_s = 0.001
duty = 0.25
mod_depth = 0.90

phases = [0, 0.125, 0.25, 0.5, 0.875]
readouts_ms = [5, 10, 15, 20, 25, 30]

rows_out = []

for readout_ms in readouts_ms:

    readout_s = readout_ms / 1000.0
    row_time = readout_s / H

    for phase_offset in phases:

        gain1 = np.zeros(H)
        gain2 = np.zeros(H)

        for r in range(H):

            exposure_offsets = np.linspace(
                0.0,
                exposure_s,
                Ns
            )

            t1 = r * row_time + exposure_offsets
            phi1 = np.mod(freq_hz * t1, 1.0)

            led1 = phi1 < duty

            intensity1 = (
                (1.0 - mod_depth)
                + mod_depth * led1.astype(float)
            )

            gain1[r] = np.mean(intensity1)

            t2 = r * row_time + exposure_offsets
            phi2 = np.mod(
                freq_hz * t2 + phase_offset,
                1.0
            )

            led2 = phi2 < duty

            intensity2 = (
                (1.0 - mod_depth)
                + mod_depth * led2.astype(float)
            )

            gain2[r] = np.mean(intensity2)

        for r in range(H):

            rows_out.append([
                readout_ms,
                phase_offset,
                r,
                gain1[r],
                gain2[r]
            ])

df = pd.DataFrame(
    rows_out,
    columns=[
        "ReadoutMs",
        "Phase",
        "Row",
        "GainFrame1",
        "GainFrame2"
    ]
)

df.to_csv(
    str(VERIFICATION_ROOT / "rowgain_python.csv"),
    index=False
)

print(f"Saved {VERIFICATION_ROOT / 'rowgain_python.csv'}")
print("Rows =", len(df))

In [ ]:
import os
import numpy as np
from PIL import Image

# ------------------------------------------------------------
# 1. FIND THE EXISTING rgb FOLDER
# ------------------------------------------------------------

possible_roots = [
    str(DATASET_SEARCH_ROOT / "rgbd_dataset_freiburg1_xyz"),
    str(DATASET_EXTRACT_ROOT / "rgbd_dataset_freiburg1_xyz"),
    str(DATASET_SEARCH_ROOT)
]

rgb_dir = None

for root in possible_roots:
    candidate = os.path.join(root, "rgb")
    if os.path.isdir(candidate):
        rgb_dir = candidate
        break

if rgb_dir is None:
    # Search recursively as fallback
    for root, dirs, files in os.walk(str(DATASET_SEARCH_ROOT)):
        if os.path.basename(root) == "rgb":
            pngs = [f for f in files if f.lower().endswith(".png")]
            if len(pngs) > 100:
                rgb_dir = root
                break

if rgb_dir is None:
    raise FileNotFoundError("Could not find the extracted rgb folder.")

print("Using rgb folder:")
print(rgb_dir)

# ------------------------------------------------------------
# 2. SELECT THE SAME 100th IMAGE
# ------------------------------------------------------------

files = sorted([
    f for f in os.listdir(rgb_dir)
    if f.lower().endswith(".png")
])

print("Number of PNG images:", len(files))

img_name = files[99]   # Python index 99 = MATLAB image 100
img_path = os.path.join(rgb_dir, img_name)

print("Using image:")
print(img_path)

img = np.array(
    Image.open(img_path).convert("RGB")
)

print("Original shape:", img.shape)

# ------------------------------------------------------------
# 3. MATCHED PWM SETTINGS
# ------------------------------------------------------------

H = img.shape[0]
Ns = 100

freq_hz = 100
exposure_s = 0.001
duty = 0.25
mod_depth = 0.90

phase_offset = 0.25
readout_ms = 20

readout_s = readout_ms / 1000.0
row_time = readout_s / H

row_gain = np.zeros(H, dtype=np.float64)

# ------------------------------------------------------------
# 4. GENERATE ROW-GAIN PROFILE
# ------------------------------------------------------------

for r in range(H):

    exposure_offsets = np.linspace(
        0.0,
        exposure_s,
        Ns
    )

    t = r * row_time + exposure_offsets

    phi = np.mod(
        freq_hz * t + phase_offset,
        1.0
    )

    led_on = phi < duty

    instantaneous_gain = (
        (1.0 - mod_depth)
        + mod_depth * led_on.astype(float)
    )

    row_gain[r] = np.mean(
        instantaneous_gain
    )

# ------------------------------------------------------------
# 5. APPLY GAIN
# ------------------------------------------------------------

perturbed = (
    img.astype(np.float64)
    * row_gain[:, None, None]
)

# Match final Python implementation
perturbed = np.clip(
    np.rint(perturbed),
    0,
    255
).astype(np.uint8)

# ------------------------------------------------------------
# 6. SAVE OUTPUTS
# ------------------------------------------------------------

out_img = str(VERIFICATION_ROOT / "perturbed_python.png")
out_gain = str(VERIFICATION_ROOT / "rowgain_single_python.csv")

Image.fromarray(perturbed).save(out_img)

np.savetxt(
    out_gain,
    row_gain,
    delimiter=","
)

print("\nSaved:")
print(out_img)
print(out_gain)
print("Image shape:", perturbed.shape)